<h2 align="center"><b>Analysis of MDS Trajectory: Binding Free energy (MMGBSA) </b></h2>

<div align="center">
  <img src="https://github.com/SHEDOOMTC/Basic_Moco_2/raw/main/Assets/Moco_2_visuals.png" alt="Description" width="400"/>
</div>

<h2 style="text-align: left; color: darkred; font-weight: bold;">About and Usage</h2>

In this tutorial, we will continue from where we stopped in the practical session **[Practicals_AmberTools_Analysis.ipynb](Link)**. Now, we will look at how to compute Binding Free Energy **[BFE]()** of ligands bound to proteins after simulation run in **[AmberTools](https://ambermd.org/AmberTools.php)**.

This is a practical session follow up of the course on **"[Basic Molecular Modelling Course for Chemists and Life Sciences II](https://github.com/SHEDOOMTC/Basic_Moco_2)** ***(Basic Moco II)***"

Cite this tutorial as **"""""**

<h3 style="text-align: left; color: darkblue; font-weight: bold;">MMGBSA</h3>

**[MM-GBSA (Molecular Mechanics/Generalized Born Surface Area)]()** is an endpoint free energy calculation method used after molecular dynamics simulation to estimate the binding affinity between a protein and a ligand. It calculates the binding free energy (ΔG bind) as the difference between the energy of the complex and the sum of the energies of the receptor and ligand, combining molecular mechanics terms with an implicit solvent model.

The binding free energy is estimated as:

$$
\Delta G_{bind} = G_{complex} - G_{receptor} - G_{ligand}
$$

**[MM-PBSA (Molecular Mechanics Poisson–Boltzmann Surface Area)]()** numerically solves the Poisson–Boltzmann equation on a 3D grid, offering higher physical rigor for large or highly charged complexes. Conversely, MM/GBSA uses an analytical Generalized Born approximation, trading exact electrostatics for dramatically faster speeds. Researchers typically favor MM/GBSA for rapid high-throughput ligand ranking and reserve MM/PBSA for complex interfaces or final validation where rigorous electrostatics outweigh computational cost.

For the computation here, we will use the trajectories from the production run of the tutorial on AmberTools "**[Practicals_AmberTools.ipynb](Link)**. This trajectory has already been processed in the tutorial on **[Practicals_AmberTools_Analysis.ipynb](Link)** and contains 10000 frames of ligand-bound **[3HTB](https://www.rcsb.org/structure/3HTB)** (T4 lysozyme).

And **[CPPTRAJ](https://doi.org/10.1021/ct400341p)** is our work engine.

In [ ]:
#If you want to skip the whole mmgbsa step, then copy the full reference analysis from ../mmgbsa_run_done  and just make the plots
#!cp -r ../mmgbsa_run_done/* .

In [ ]:
#Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nglview as nv

<h3 style="text-align: left; color: darkblue; font-weight: bold;">The Trajectory</h3>

The accuracy of MMGBSA is tied to its ability to derive its energy from an ensemble average by using many frames from the trajectory of our simulation.

In [ ]:
# Copy the processed trajectory from the previous tutorial:
#You can copy from your own run or from the reference folder provided in the tutorial.
#!cp ../AmberTools_Analysis/traj_strip.nc && cp ../AmberTools_Analysis/stripped.complex.top .

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Complex, Ligand and Protein Topologies</h3>

Since MMGBSA computes the difference between the energy of the complex and the sum of the energies of the receptor and ligand, we must futher process our system (or complex) topology to extract only the topologies of the ligand, and protein separately

- **Ante-MMPBSA.py**

**[Ante-MMPBSA.py](Link)** is a Python utility in AmberTools that automates and streamlines the preparation of molecular topology (prmtop) files for MM/PBSA and MM/GBSA free energy calculations. It helps to strip solvents and ions to ensure dry topologies for implicit-solvent calculations, split complexes to generate separate topology files for the complex (-c), receptor (-r), and ligand (-l) based on user-defined residue or atom masks. It also allows users to adjust intrinsic implicit solvent radius sets (such as mbondi, mbondi2, or bondi) to match the requirements of the chosen Poisson-Boltzmann (PB) or Generalized Born (GB) model.

Ante-MMPBSA.py does not strip trajectories of water and ions. This was already done using CPPTRAJ in the last analysis.

So, let us run it on our topology.

In [ ]:
!ante-MMPBSA.py -p stripped.complex.top -c com.top -r rec.top -l JZ4.top -s :WAT:Na+:Cl- -n :164 --radii=mbondi2

The *-s* flag allow us to specify the heteroatoms to remove during the stripping. the *-n* flag specify the ligand residue index, from where to generate separate ligand and protein topologies. In the above com.top is still the same complex topology; rec.top is the protein-only topology, while JZ4.top. We specify the *mbondi2* as our implicit solvent radius 

<font color="darkred">*Since our trajectories already had water and ions stripped, did we need the *-s* flag?*</font>

<h3 style="text-align: left; color: darkblue; font-weight: bold;">The MMGBSA Input File</h3>

The input file is generally written in fortran. It starts with a namelist precedded by *&*. Popular namelists are the *&general, &gb, &pb, and &decomp*. 

The *&general* sets the global simulation and trajectory controls such as trajectory frames to read (startframe, endframe, interval), specifies whether to strip solvent on-the-fly (strip_mask), and dictates file management behaviors (keep_files, verbose).

The *&gb* controls the Generalized Born (GB) implicit solvent calculations while the *&pb* controls Poisson-Boltzmann (PB) implicit solvent calculations.

The *&decomp* breaks down total energy scores into individual components per residue **(Per-Residue Decomposition)**.It splits the binding energy into specific contributions from individual amino acids or base pairs. It highlights which specific parts of a receptor or ligand are driving the binding interactions using schemes like 1-4 interaction addition or subtraction (idecomp).

In [ ]:
%%bash

# Lets make a mmgbsa.in file
cat > mmgbsa.in << EOF
&general
 startframe=1,  endframe=10000, interval=50, keep_files=0,
/

&gb
igb=5,saltcon=0.15,
/

#!&pb
#!istrng=0.100,
#!/

&decomp
idecomp=1, 
print_res="164;78;84;87;88;91;99;100;102;103;111;114;117;118;121;133;153,"
dec_verbose=1,csv_format=0,
/
run
EOF

We will use only **200 frames** that are well spaced from our trajectories, so our interval will be 50. This choice is to reduce the computational cost. 200 equally spaced frames should represent distinct configurations sampled during the simulation. Setting *keep_files* flag to zero ensures that all intermediate files generated during the run are removed automatically (to avoid clutering your workspace). 

<font color="darkred">*Why did we comment out the *&pb* namelist?*</font>

<font color="darkred">*What do you think about the computational cost of running the mmgbsa and mmpbsa models?*</font>

In the per-residue decomposition; we will look at the contribution of the resideus in the active site to the BFE. These residues include *:78,87,91,100,103,111,118,133,153*

<h3 style="text-align: left; color: darkblue; font-weight: bold;">The MMGBSA Run</h3>

- **MMPBSA.py**

**[MMPBSA.py](Link)** is the primary Python script in the AmberTools suite used to calculate binding free energies for molecular complexes. It handles both MM/GBSA (Generalized Born) and MM/PBSA (Poisson-Boltzmann) end-state free energy calculations.

- **Let us run it**

In [ ]:
!mpirun -np 4 MMPBSA.py -O -i mmgbsa.in -o mmgbsa.dat -sp stripped.complex.top -cp com.top -rp rec.top -lp JZ4.top -y traj_strip.nc > mmgbsa.log

The main energy file is stores in the *mmgbsa.dat file*, while another file *FINAL_DECOMPOSITION.dat* will also be generated since we have the *&idecomp* namelist on. The log file helps us to track the run and check for error calls incase of abrupt termination.

### **5. Analysis**
<h3 style="text-align: left; color: darkblue; font-weight: bold;">SOLVATION MODELS</h3>

- **The mmgbsa.dat file**

The output file has three sections: *the complex energy components*, *the receptor energy component*, *the ligand energy component* and *the difference component*. The difference is the most important since this is the true BFE.

This energy is decomposed into the various functional forms such as the van der Waals **(VDWAALS)**, electrostatic **(EEL)**, polar solvation energy **(EGB)** non-polar solvation energy **(ESURF)**, binding free energy in the gas phases **(DELTA Ggas)**, total solvation energy **(DELTA Gsolv)** and the final BFE **(DELTA TOTAL)**

In [ ]:
!cat mmgbsa.dat

We will extract these energies into a file and make a plot. There are two python scripts that is available in the reference folder which will help us do this

In [ ]:
#Copy the Extract_mmgbsa.py and Plot_mmgbsa.py from the plot_scripts folder
!cp ../plots_script/Extract_mmgbsa.py .
!cp ../plots-scripts/Plot_mmgbsa.py .

# 1. Import the function from your script
from Extract_mmgbsa import process_mmgbsa
from Plot_mmgbsa import plot_mmgbsa

# 2. Pass your file into it
df_energy = process_mmgbsa("mmgbsa.dat")
df_energy 

# 3. Plot the results
plot_mmgbsa(df_energy)

<font color="darkred">*How does electrostatic and van der Waal forces influnce the total binding energy of JZ4 to our sytem?*</font>

- **The Per Residue Decomposition File**

Per-residue energy decomposition breaks down a molecular complex's total binding free energy into individual contributions from each amino acid. By isolating van der Waals, electrostatic, and solvation energies for every residue, it identifies critical "hot spots" driving the interaction.

In [ ]:
!cat FINAL_DECOMPOSITION.dat

We will extract these energies into a file and make a plot. There are two python scripts that is available in the reference folder which will help us do this

In [ ]:
#Copy the Extract_decomp.py and Plot_decomp.py from the reference folder
!cp ../Extract_decomp.py .
!cp ../Plot_decomp.py .

#1. Import the function
from Extract_decomp import process_decomp
from Plot_decomp import plot_decomp

#2. Pass your file into it
df_decomp = process_decomp("FINAL_DECOMP_MMPBSA_CW044.dat")
df_decomp
# 2. Generate the plot
plot_decomp(df_decomp)

<font color="darkred">*Which residues had the greatest contribution of energy to the total energy*</font>

<font color="darkred">*Are these energies truly additive?*</font>

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Summary and Reflections</h3>

The Molecular Mechanics Generalized Born Surface Area (MM/GBSA) protocol estimates binding free energy from a molecular dynamics trajectory. It calculates the gas-phase energy (van der Waals and electrostatics via molecular mechanics) and subtracts solvation energies. Solvation combines polar contributions via the Generalized Born model and non-polar contributions via a Surface Area term. 

By comparing the energies of the complex, receptor, and ligand, the protocol computes the binding free energy and per-residue decomposition. This post-processing method offers a computationally efficient way to rank binding affinities and identify key interacting residues compared to more rigorous free energy methods.

<h3 style="text-align: left; color: darkblue; font-weight: bold;">References</h3>

- https://ambermd.org/doc12/Amber25.pdf
- https://ambermd.org/tutorials/
- https://doi.org/10.1517/17460441.2015.1032936
- https://github.com/SHEDOOMTC/AmberTools-Analysis-Quick.git